# Backgammon Dataset Playground

Notebook for exploring `ArkadiumInc/ArkadiumBackgammon`.

In [1]:
from pathlib import Path

from datasets import load_dataset

# Make paths robust whether notebook runs from repo root or notebooks/
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

In [2]:
# Load game logs only (no images)
ds = load_dataset("ArkadiumInc/ArkadiumBackgammon", "gamelogs")

In [3]:
match = ds["train"][0]

print(f"Players: {match['player1']} vs {match['player2']}")
print(f"Date: {match['event_date']}")
print(f"Result: {match['result']}")
print(f"\nMoves:\n{match['moves']}")

Players: pl1 vs pl2
Date: 2025.06.05
Result: pl1+6

Moves:
1) 35: 8/3 6/3                 43: 24/20 24/21
2) 61: 13/7 7/6                26: 21/15 15/13
3) 33: 8/5* 8/5 24/21 24/21    54: 25/21 13/8
4) 46: 13/7 7/3                45: 13/8 21/17
5) 31: 6/5 21/18               22: 17/15 15/13 13/11 13/11
6) 61: 13/7 21/20               Doubles => 2
7) Takes                      14: 11/7* 8/7
8) 46: 25/21 20/14*            43: 25/21 21/18*
9) 11: 25/24 14/13 5/4 6/5     45: 8/3 8/4*
10) 14: 25/21* 5/4              42: 25/23 13/9
11) 61: 13/7* 7/6               33:
12) 35: 21/16* 16/13            43:
13) 11: 24/23 23/22* 22/21


In [4]:
ds["train"].to_pandas().head()

,data_id,file_id,place,player1,player2,event_date,event_time,cube_limit,match_length,game_uuid,result,moves,raw_log
0,bg_1200001,1200001,Arkadium,pl1,pl2,2025.06.05,15.46,64,3,987b847a-9ed5-9e0e-1ef4-b46757caeb39,pl1+6,1) 35: 8/3 6/3 43: 24/20 24/21...,"; [Place ""Arkadium""]\n; [Player 1 ""pl1""]\n; [P..."
1,bg_1200002,1200002,Arkadium,pl1,pl2,2025.06.13,21.02,1,1,7c07cd22-fb29-736f-591a-b7f0a55922bf,pl1+2,1) 12: 13/11 24/23\n2) 53: 8/3 6/3 ...,"; [Place ""Arkadium""]\n; [Player 1 ""pl1""]\n; [P..."
2,bg_1200003,1200003,Arkadium,pl1,pl2,2025.05.08,20.16,64,5,86b597d2-a2d4-51cc-3f5b-3a83546e01d9,void,1) 34: 13/9 9/6\n2) 33: 24/21 24/21 21/18 21/1...,"; [Place ""Arkadium""]\n; [Player 1 ""pl1""]\n; [P..."
3,bg_1200004,1200004,Arkadium,pl1,pl2,2025.06.13,21.06,1,1,0c767f6f-9aa6-1169-a913-4e68f4927a21,pl1+1,1) 54: 24/20 13/8 15: 13/8 24/23\...,"; [Place ""Arkadium""]\n; [Player 1 ""pl1""]\n; [P..."
4,bg_1200005,1200005,Arkadium,pl1,pl2,2025.06.03,15.00,1,1,c5b8a23f-32cb-22f9-7505-82896a36b352,pl1+2,1) 43: 13/10 10/6 22: 13/11 13/11...,"; [Place ""Arkadium""]\n; [Player 1 ""pl1""]\n; [P..."


In [5]:
from pathlib import Path
import subprocess
import sys

project_root = Path.cwd().resolve()
if project_root.name == "notebooks":
    project_root = project_root.parent

script_path = project_root / "scripts" / "train_moves_lm.py"
mat_zip_path = project_root / "data" / "Arkadium_Backgammon_full_data_gamelogs_001.zip"

cmd = [sys.executable, str(script_path)]
if mat_zip_path.exists():
    cmd += ["--mat-zip", str(mat_zip_path), "--max-samples", "0"]
else:
    print(f"Warning: {mat_zip_path} not found; using HF preview dataset.")
    cmd += ["--max-samples", "20000"]

cmd += [
    "--seq-len",
    "96",
    "--epochs",
    "2",
    "--batch-size",
    "8",
    "--holdout-ratio",
    "0.2",
]
subprocess.run(cmd, check=True, cwd=str(project_root))


  0%|          | 0/2 [00:00<?, ?it/s]/Users/benjaminlaufer/Python Projects/backgammon-analysis/.venv/lib/python3.10/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.
 50%|█████     | 1/2 [00:00<00:00,  1.85it/s]
                                             
100%|██████████| 1/1 [00:00<00:00, 864.98it/s]
                                              
Writing model shards: 100%|██████████| 1/1 [00:00<00:00, 21.29it/s]
/Users/benjaminlaufer/Python Projects/backgammon-analysis/.venv/lib/python3.10/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
100%|██████████| 2/2 [00:00<00:00,  2.62it/s

{'eval_loss': '5.317', 'eval_runtime': '0.0909', 'eval_samples_per_second': '22.01', 'eval_steps_per_second': '11.01', 'epoch': '1'}
{'eval_loss': '5.25', 'eval_runtime': '0.0113', 'eval_samples_per_second': '177', 'eval_steps_per_second': '88.51', 'epoch': '2'}


Writing model shards: 100%|██████████| 1/1 [00:00<00:00, 41.77it/s]
/Users/benjaminlaufer/Python Projects/backgammon-analysis/.venv/lib/python3.10/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
100%|██████████| 1/1 [00:00<00:00, 1901.32it/s]


{'train_runtime': '0.9397', 'train_samples_per_second': '17.03', 'train_steps_per_second': '2.128', 'train_loss': '5.42', 'epoch': '2'}
Final eval metrics: {'eval_loss': 5.250483989715576, 'eval_runtime': 0.0098, 'eval_samples_per_second': 204.685, 'eval_steps_per_second': 102.343, 'epoch': 2.0}
Saved model and tokenizer to: artifacts/moves-lm/final
Saved unseen holdout rows to: artifacts/moves-lm/holdout_rows.jsonl


CompletedProcess(args=['/Users/benjaminlaufer/Python Projects/backgammon-analysis/.venv/bin/python', '/Users/benjaminlaufer/Python Projects/backgammon-analysis/scripts/train_moves_lm.py', '--max-samples', '20000', '--seq-len', '96', '--epochs', '2', '--batch-size', '8', '--holdout-ratio', '0.2'], returncode=0)

In [6]:
from pathlib import Path

import pandas as pd
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

project_root = Path.cwd().resolve()
if project_root.name == "notebooks":
    project_root = project_root.parent

model_dir = project_root / "artifacts" / "moves-lm" / "final"
holdout_path = project_root / "artifacts" / "moves-lm" / "holdout_rows.jsonl"

if not model_dir.exists():
    raise FileNotFoundError(f"Model not found at {model_dir}. Run training first.")

if holdout_path.exists():
    holdout_df = pd.read_json(holdout_path, lines=True)
else:
    # Fallback: recreate the same holdout split in-memory for notebook robustness.
    ds_all = load_dataset("ArkadiumInc/ArkadiumBackgammon", "gamelogs")["train"]
    ds_all = ds_all.filter(lambda x: isinstance(x["moves"], str) and len(x["moves"].strip()) > 0)
    split = ds_all.train_test_split(test_size=0.2, seed=42)
    holdout_df = split["test"].to_pandas()
    print(f"Warning: {holdout_path} not found; using in-memory holdout split instead.")

tokenizer = AutoTokenizer.from_pretrained(str(model_dir), local_files_only=True)
model = AutoModelForCausalLM.from_pretrained(str(model_dir), local_files_only=True)

device = "mps" if torch.backends.mps.is_available() else "cpu"
model = model.to(device)

# Select a row from the unseen holdout split
row_idx = 0
full_moves = holdout_df.iloc[row_idx]["moves"]

# Use prefix prompt from that unseen row
prompt_tokens = full_moves.split()[:20]
prompt = " ".join(prompt_tokens)

inputs = tokenizer(prompt, return_tensors="pt").to(device)
with torch.no_grad():
    out = model.generate(
        **inputs,
        max_new_tokens=30,
        do_sample=True,
        temperature=0.8,
        top_p=0.95,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

generated = tokenizer.decode(out[0], skip_special_tokens=True)

print("HOLDOUT ROW INDEX:", row_idx)
print("PROMPT:\n", prompt)
print("\nMODEL CONTINUATION:\n", generated)
print("\nGROUND TRUTH (full unseen row):\n", full_moves)

/var/folders/35/rbf5cr3s6wx53bzp489khqh40000gn/T/ipykernel_79332/1262498752.py:19: FutureWarning: The behavior of 'to_datetime' with 'unit' when parsing strings is deprecated. In a future version, strings will be parsed as datetime strings, matching the behavior without a 'unit'. To retain the old behavior, explicitly cast ints or floats to numeric type before calling to_datetime.
  holdout_df = pd.read_json(holdout_path, lines=True)


Loading weights:   0%|          | 0/52 [00:00<?, ?it/s]

HOLDOUT ROW INDEX: 0
PROMPT:
 1) 41: 13/9 9/8 2) 13: 8/5 6/5 33: 8/5 8/5 24/21 24/21 3) 34: 24/21 6/2 33: 13/10 13/10

MODEL CONTINUATION:
 1) 41: 13/9 9/8 2) 13: 8/5 6/5 33: 8/5 8/5 24/21 24/21 3) 34: 24/21 6/2 33: 13/10 13/10 25/24 13/8 15: 6/1 13/7 6/3 13/8 13/8 21: 21: 13/7 12) 20/15 6/5 6/3 21/20 54: 12)

GROUND TRUTH (full unseen row):
 1) 41: 13/9 9/8
2) 13: 8/5 6/5                 33: 8/5 8/5 24/21 24/21
3) 34: 24/21 6/2               33: 13/10 13/10 10/7 10/7
4) 45: 13/8 6/2                46: 13/7 13/9
5) 33: 24/21 13/10 13/10 8/5   36: 9/3 6/3
6) 51: 13/8 13/12              52: 21/16 16/14
7) 11: 12/11* 5/4* 4/3 3/2     42: 25/21
8) 56: 21/15 21/16             13: 25/22 22/21
9) 55: 16/11 15/10 11/6 11/6   44: 8/4 8/4 6/2 6/2
10) 35: 10/5 8/5                62: 7/1 3/1
11) 54: 6/1 5/1                 61: 7/1 4/3
12) 63: 8/2 8/5                 53: 7/2 4/1
13) 42: 5/1 5/3                 25: 21/16 16/14
14) 36: 10/4* 6/3               46: 25/21* 21/15*
15) 46: 25/21          

In [9]:
holdout_df

,data_id,file_id,place,player1,player2,event_date,event_time,cube_limit,match_length,game_uuid,result,moves,raw_log
0,bg_1200006,1200006,Arkadium,pl1,pl2,2025.06.04,1970-01-01 00:00:09.010,1,1,767ac505-0cca-1121-63bb-dbba229b84a7,pl1+1,1) 41: 13/9 9/8\n2) 13: 8/5 6/5 ...,"; [Place ""Arkadium""]\n; [Player 1 ""pl1""]\n; [P..."
1,bg_1200007,1200007,Arkadium,pl1,pl2,2025.05.13,1970-01-01 00:00:10.530,1,1,c7f61387-9bdb-9304-eac1-3bebaddffe22,pl1+1,1) 31: 8/5 6/5\n2) 21: 13/11 11/10 ...,"; [Place ""Arkadium""]\n; [Player 1 ""pl1""]\n; [P..."
